In [ ]:
from skimage.registration import phase_cross_correlation
import xarray as xr
import rioxarray
from pathlib import Path
import numpy as np

In [32]:
root = Path.cwd()
p1 = rioxarray.open_rasterio(root / 'model_outputs' / 'nyc18_from_nyc21_medians_only_model_canopy_percent.tif')
p2 = rioxarray.open_rasterio(root / 'model_outputs' / 'nyc21_from_nyc21_medians_only_model_canopy_percent.tif')

l1 = rioxarray.open_rasterio(root / 'data' / 'nyc18' / 'nyc18_tree_canopy_lidar.tif')
l2 = rioxarray.open_rasterio(root / 'data' / 'nyc21' / 'nyc21_tree_canopy_lidar.tif')


p1 = p1.fillna(0)
l1 = l1.fillna(0)
p2 = p2.fillna(0)
l2 = l2.fillna(0)


In [15]:
shift, error, diffphase = phase_cross_correlation(l2, l1, upsample_factor=20)
print(f'shift: {shift}\nerror: {error}\ndiffphase: {diffphase}')

shift: [0. 0. 0.]
error: 1.0
diffphase: -2.968815471238173e-11


In [41]:
p2 = rioxarray.open_rasterio(root / 'model_outputs' / 'nyc21_from_nyc21_medians_only_model_canopy_percent.tif')
p2 = p2.fillna(0)
s, error, diffphase = phase_cross_correlation(l2.squeeze('band'),p2.squeeze('band'),  upsample_factor=20)
print(f'shift: {shift}\nerror: {error}\ndiffphase: {diffphase}')

shift: [0.05 0.2 ]
error: 1.0
diffphase: -6.43381426925771e-05


In [ ]:
## 2021 prediction raster has ~2m shift from other three rasters. 

## for some reason, 2021 tc layer was not aligned to 2021 satellite data. 

In [44]:
from scipy.ndimage import shift

p2 = rioxarray.open_rasterio(root / 'model_outputs' / 'nyc21_from_nyc21_medians_only_model_canopy_percent.tif')
p2 = p2.fillna(0)
s, error, diffphase = phase_cross_correlation(l2.squeeze('band'),p2.squeeze('band'),  upsample_factor=20)
print(f'shift: {s}\nerror: {error}\ndiffphase: {diffphase}')


dy, dx = s
p2 = rioxarray.open_rasterio(root / 'model_outputs' / 'nyc21_from_nyc21_medians_only_model_canopy_percent.tif').squeeze('band')
p2_shift = shift(p2, [dy, dx], order=1)

p2_shift_fillna = p2_shift.copy()
p2_shift_fillna[~np.isfinite(p2_shift_fillna)] = 0

s, error, diffphase = phase_cross_correlation(l2.squeeze('band'),p2_shift_fillna,  upsample_factor=20)
print(f'shift: {s}\nerror: {error}\ndiffphase: {diffphase}')

shift: [0.05 0.2 ]
error: 1.0
diffphase: -6.43381426925771e-05
shift: [0.05 0.1 ]
error: 1.0
diffphase: -4.023583824164234e-05


shift: [-0.1 -0.3]
error: 1.0
diffphase: 9.4104754680302e-05
